# Normwise and componentwise conditioning

The conditioning notebook measured a solve against κ(A)·u, the normwise
condition number times the unit roundoff. That number answers a particular
question: how far can x move when A and b are perturbed by a small amount
*relative to their norms*? Rounding errors are not like that. Rounding an
entry perturbs it relative to *that entry*, so a tiny entry gets a tiny
perturbation. For a badly scaled matrix the two questions have very different
answers, and κ(A) can overstate the attainable error by many orders of
magnitude.

The componentwise theory (Bauer, Skeel, Oettli and Prager; Higham,
*Accuracy and Stability of Numerical Algorithms*, ch. 7) measures
perturbations entry by entry:

- the **componentwise backward error** of a computed x̂ is the smallest ω with
  (A + ΔA) x̂ = b + Δb, |ΔA| ≤ ω|A| and |Δb| ≤ ω|b|. Oettli and Prager showed
  it is

  $$\omega(\hat x) = \max_i \frac{|b - A\hat x|_i}{(|A|\,|\hat x| + |b|)_i};$$

- the matching **condition number** of the solution x is

  $$\operatorname{cond}(A, x) = \frac{\lVert\, |A^{-1}|\,(|A|\,|x| + |b|)\,\rVert_\infty}{\lVert x\rVert_\infty},$$

  so that, to first order in ω, ‖x̂ − x‖∞/‖x‖∞ ≤ cond(A, x) · ω;
- **Skeel's condition number** cond(A) = ‖ |A⁻¹| |A| ‖∞ satisfies
  cond(A, x) ≤ 2 cond(A) for every x, since |b| ≤ |A||x|. Unlike κ, it
  doesn't change when the rows of A are scaled.

This notebook:

1. **Scales the rows** of a well-conditioned matrix: κ∞ grows to 10¹⁸ while
   cond(A) stays at 47, and the error of an LU solve follows cond, not κ.
2. **Shows one step of iterative refinement** in working precision bringing
   the componentwise backward error of that solve back to u (Skeel, 1980).
3. **Scales the columns**: κ∞ grows as well, but only the units of x change.
   Each component of x keeps its relative accuracy, while the normwise error,
   dominated by the largest components, says nothing about the smallest.
4. **Repeats the row scaling in each format**, where it stops being harmless:
   fp16 flushes small entries to zero, and posits and takums lose digits as
   the entries move away from 1.

Every random matrix has a seed, so the figures are the same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

FAMILY = {"IEEE": SERIES[0], "posit": SERIES[1], "takum": SERIES[2]}
WIDTHS = {
    "16 bits": [("fp16", "IEEE"), ("posit16", "posit"), ("takum16", "takum")],
    "32 bits": [("f32", "IEEE"), ("posit32", "posit"), ("takum32", "takum")],
    "64 bits": [("f64", "IEEE")],
}
U64 = 2.0**-53

N = 24
B = g.randsvd(N, N, 10.0, 3, seed=11).to_numpy()  # kappa_2(B) = 10
rng = np.random.default_rng(5)
PERM = rng.permutation(N)
X_TRUE = rng.standard_normal(N)


def as_array(v):
    # Always float64: f32's to_numpy() returns float32.
    return np.asarray(v.to_numpy() if hasattr(v, "to_numpy") else v, dtype=np.float64)


def scaling(s):
    # Powers of 2 from 1 down to 2^-s, in a seeded random order. Scaling by a
    # power of 2 is exact in binary arithmetic, so the scaled matrices differ
    # from B by the scaling alone, not by rounding.
    return np.ldexp(1.0, -np.round(np.linspace(0, s, N)).astype(int))[PERM]


def reference(A, b):
    # A quad-double solve of the stored float64 system, as the exact solution.
    return as_array(mtl5.solve(mtl5.convert(A, "qd_cascade"), mtl5.convert(b, "qd_cascade")))


def residual(A, x, b):
    # b - A x with each component computed exactly and rounded once, so the
    # backward errors below are not limited by a float64 residual.
    one = np.array([1.0])
    return np.array(
        [
            mtl5.mixed.dot(
                mtl5.vector(np.concatenate([[b_i], -a_i])),
                mtl5.vector(np.concatenate([one, x])),
                accumulator="exact",
            )
            for a_i, b_i in zip(A, b)
        ]
    )


def omega(A, x, b):
    # Componentwise backward error (Oettli-Prager).
    return np.max(np.abs(residual(A, x, b)) / (np.abs(A) @ np.abs(x) + np.abs(b)))


def conditions(A, x, b):
    Ai = np.linalg.inv(A)
    kappa = np.linalg.norm(A, np.inf) * np.linalg.norm(Ai, np.inf)
    skeel = np.linalg.norm(np.abs(Ai) @ np.abs(A), np.inf)
    cond_x = np.linalg.norm(
        np.abs(Ai) @ (np.abs(A) @ np.abs(x) + np.abs(b)), np.inf
    ) / np.linalg.norm(x, np.inf)
    return kappa, skeel, cond_x


def normwise_error(y, x):
    return np.linalg.norm(y - x, np.inf) / np.linalg.norm(x, np.inf)


def componentwise_error(y, x):
    return np.max(np.abs(y - x) / np.abs(x))


def lu_and_refine(A, b):
    # LU with partial pivoting in float64, then one step of fixed-precision
    # iterative refinement: the residual in float64, the correction from the
    # same factors.
    F = mtl5.lu(mtl5.matrix(A))
    x0 = as_array(F.solve(b))
    x1 = x0 + as_array(F.solve(b - A @ x0))
    return x0, x1

## 1. Row scaling: κ grows, cond doesn't

The base matrix B is `randsvd(24, 24, kappa=10, mode=3, seed=11)`. Each sweep
point scales its rows by D = diag(2⁰, …, 2⁻ˢ) in a seeded random order and
solves (DB) x = Db for a fixed seeded x, with s from 0 to 60. Row scaling
doesn't change the solution, so the problem is the same at every s, only
written in different units per equation.

LU with partial pivoting (`mtl5.lu`) solves it in float64. The figure shows
the three condition numbers and the measured relative forward error divided by
u, before and after one step of iterative refinement.

In [ ]:
S_ROW = np.arange(0, 61, 4)
rows = []
for s in S_ROW:
    A = scaling(s)[:, None] * B
    b = A @ X_TRUE
    x = reference(A, b)
    x0, x1 = lu_and_refine(A, b)
    kappa, skeel, cond_x = conditions(A, x, b)
    rows.append(
        dict(
            s=s,
            kappa=kappa,
            skeel=skeel,
            cond_x=cond_x,
            err0=normwise_error(x0, x),
            err1=normwise_error(x1, x),
            om0=omega(A, x0, b),
            om1=omega(A, x1, b),
        )
    )
col = {k: np.array([r[k] for r in rows]) for k in rows[0]}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.semilogy(
    S_ROW, col["kappa"], color=SERIES[0], linewidth=2, marker="o", markersize=4, label="κ∞(A)"
)
ax.semilogy(
    S_ROW,
    col["skeel"],
    color=SERIES[1],
    linewidth=2,
    marker="s",
    markersize=4,
    label="cond(A), Skeel",
)
ax.semilogy(
    S_ROW, col["cond_x"], color=SERIES[2], linewidth=2, marker="^", markersize=4, label="cond(A, x)"
)
ax.set_xlabel("s, rows scaled down to 2⁻ˢ")
ax.set_title("Condition numbers under row scaling", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")

ax = axes[1]
ax.semilogy(
    S_ROW, col["err0"] / U64, color=SERIES[0], linewidth=2, marker="o", markersize=4, label="LU"
)
ax.semilogy(
    S_ROW,
    col["err1"] / U64,
    color=SERIES[1],
    linewidth=2,
    marker="s",
    markersize=4,
    label="LU + one refinement step",
)
ax.semilogy(
    S_ROW,
    col["kappa"],
    color=RULE,
    linestyle="--",
    linewidth=1.5,
    label="κ∞(A), the normwise scale",
)
ax.set_ylim(1e-1, 1e20)
ax.set_xlabel("s, rows scaled down to 2⁻ˢ")
ax.set_ylabel("relative forward error / u")
ax.set_title("Measured error in float64", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

print("   s   kappa_inf  cond(A)  cond(A,x)  error/u (LU, refined)  omega/u (LU, refined)")
for r in rows:
    print(
        f"{r['s']:4d}  {r['kappa']:9.1e}  {r['skeel']:7.1f}  {r['cond_x']:9.1f}  "
        f"{r['err0'] / U64:9.1f} {r['err1'] / U64:9.1f}     "
        f"{r['om0'] / U64:9.1f} {r['om1'] / U64:9.1f}"
    )

κ∞ grows with the scaling, by about 2ˢ, to 10¹⁸ at s = 60. If it described
the solve, f64 would have no correct digits left from s ≈ 52 on. Instead,
cond(A) and cond(A, x) are the same at every s: neither changes under row
scaling. The forward error stays below 30u, many orders of magnitude below
κ∞·u: the dashed line is that normwise scale, in units of u.

## 2. One step of refinement restores componentwise stability

LU with partial pivoting is normwise backward stable, but not componentwise:
on these row-scaled matrices its componentwise backward error ω, printed in the
last two columns above, grows from 2.4u to as much as 11.8u as the scaling
grows. The forward error grows too, to at most 27u. One step of iterative refinement, with the
residual in the *same* precision and the correction from the same LU factors,
brings ω back to about u. Skeel (1980) proved this: one step of
fixed-precision refinement makes LU with partial pivoting componentwise
backward stable, unless A is too ill conditioned or |A||x| too badly
behaved.

The figure below shows ω before and after the step.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.semilogy(
    S_ROW, col["om0"] / U64, color=SERIES[0], linewidth=2, marker="o", markersize=4, label="LU"
)
ax.semilogy(
    S_ROW,
    col["om1"] / U64,
    color=SERIES[1],
    linewidth=2,
    marker="s",
    markersize=4,
    label="LU + one refinement step",
)
ax.axhline(1, color=RULE, linestyle=":", linewidth=1)
ax.set_xlabel("s, rows scaled down to 2⁻ˢ")
ax.set_ylabel("ω / u")
ax.set_title("Componentwise backward error in float64", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

print(f"omega/u, LU:      {(col['om0'] / U64).min():.1f} to {(col['om0'] / U64).max():.1f}")
print(f"omega/u, refined: {(col['om1'] / U64).min():.1f} to {(col['om1'] / U64).max():.1f}")

## 3. Column scaling changes the units of x

Scaling the columns, A = B D, is a change of variables: the solution of
(BD) x = b is x = D⁻¹ y, where B y = b. Here y is the fixed seeded vector, so
the components of x range over s binary orders of magnitude.

Partial pivoting compares entries within a column, and scaling a column by a
power of 2 doesn't change those comparisons, so LU picks the same pivots and
every computed quantity is scaled exactly. The computed x̂ is therefore
exactly D⁻¹ times the computed ŷ, and each component has the same *relative*
error at every s. The normwise error, measured against ‖x‖∞, is set by the
largest components and says nothing about the small ones.

In [ ]:
S_COL = np.arange(0, 61, 4)
cw = []
y = X_TRUE
for s in S_COL:
    d = scaling(s)
    A = B * d[None, :]
    b = B @ y  # the same right-hand side at every s
    x = reference(A, b)
    x0, _ = lu_and_refine(A, b)
    kappa, skeel, cond_x = conditions(A, x, b)
    cw.append(
        dict(
            s=s,
            kappa=kappa,
            skeel=skeel,
            cond_x=cond_x,
            norm_err=normwise_error(x0, x),
            comp_err=componentwise_error(x0, x),
            small_err=np.abs(x0 - x)[np.argmin(np.abs(x))] / np.abs(x).min(),
        )
    )
cc = {k: np.array([r[k] for r in cw]) for k in cw[0]}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.semilogy(
    S_COL, cc["kappa"], color=SERIES[0], linewidth=2, marker="o", markersize=4, label="κ∞(A)"
)
ax.semilogy(
    S_COL,
    cc["skeel"],
    color=SERIES[1],
    linewidth=2,
    marker="s",
    markersize=4,
    label="cond(A), Skeel",
)
ax.semilogy(
    S_COL, cc["cond_x"], color=SERIES[2], linewidth=2, marker="^", markersize=4, label="cond(A, x)"
)
ax.set_xlabel("s, columns scaled down to 2⁻ˢ")
ax.set_title("Condition numbers under column scaling", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")

ax = axes[1]
ax.semilogy(
    S_COL,
    cc["norm_err"] / U64,
    color=SERIES[0],
    linewidth=2,
    marker="o",
    markersize=4,
    label="normwise: ‖x̂ − x‖∞ / ‖x‖∞",
)
ax.semilogy(
    S_COL,
    cc["comp_err"] / U64,
    color=SERIES[1],
    linewidth=2,
    marker="s",
    markersize=4,
    label="componentwise: maxᵢ |x̂ᵢ − xᵢ| / |xᵢ|",
)
ax.set_ylim(1e-1, 1e3)
ax.set_xlabel("s, columns scaled down to 2⁻ˢ")
ax.set_ylabel("relative error / u")
ax.set_title("Measured error in float64", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

print("   s   kappa_inf   cond(A)  cond(A,x)  normwise/u  componentwise/u")
for r in cw:
    print(
        f"{r['s']:4d}  {r['kappa']:9.1e}  {r['skeel']:8.1e}  {r['cond_x']:9.1e}  "
        f"{r['norm_err'] / U64:10.2f}  {r['comp_err'] / U64:15.2f}"
    )

The componentwise error is the same at every s, to the last bit, because the
computation is the same up to exact scaling. κ∞, cond(A) and cond(A, x) all
grow: none of them is invariant under column scaling, since each measures the
error relative to ‖x‖∞, and ‖x‖∞ is now set by the largest components alone.

The normwise error stays small too, at most 35u, but it is measured against
the largest components of x, the ones the scaling enlarged most. It can't
tell you how accurate the smallest are. The componentwise error can: 193u,
set by the smallest component of y, at every s, so every component of x has
the accuracy it had in the unscaled problem.

## 4. Row scaling in each format

In float64 the row scaling was harmless. The same sweep in the other formats
of #166 shows where it isn't. The scaling is still exact in binary, but only
if the format can hold the scaled entries:

- **fp16** has a narrow range. Below 2⁻¹⁴ it loses digits in its subnormals,
  and below 2⁻²⁴ it flushes entries to zero. Once a whole row is zero, A is
  singular.
- **posits and takums** hold every scaled entry, but with fewer digits the
  further it is from 1, so the scaling *does* round the matrix, more for
  larger s.

The reference solution is the same quad-double solve of the scaled float64
system as before. A solve that returns non-finite values, or an error of 1 or
more, has no correct digits.

In [ ]:
S_FMT = np.arange(0, 49, 4)
fmt_err = {name: [] for group in WIDTHS.values() for name, _ in group}
fp16_zeros = []
for s in S_FMT:
    A = scaling(s)[:, None] * B
    b = A @ X_TRUE
    x = reference(A, b)
    fp16_zeros.append(int(np.sum(as_array(mtl5.convert(A, "fp16")) == 0)))
    for name in fmt_err:
        try:
            xh = as_array(mtl5.solve(mtl5.convert(A, name), mtl5.convert(b, name)))
            e = normwise_error(xh, x) if np.all(np.isfinite(xh)) else np.inf
        except (RuntimeError, ValueError):
            e = np.inf
        fmt_err[name].append(min(e, 1.0))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (title, group) in zip(axes, WIDTHS.items()):
    for name, family in group:
        ax.semilogy(
            S_FMT,
            fmt_err[name],
            color=FAMILY[family],
            linewidth=2,
            marker="o",
            markersize=4,
            label=name,
        )
    ax.set_ylim(1e-17, 2)
    ax.set_xlabel("s, rows scaled down to 2⁻ˢ")
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8, loc="center right")
axes[0].set_ylabel("relative forward error (capped at 1)")
fig.tight_layout()
plt.show()

print("   s " + "".join(f"{name:>9s}" for name in fmt_err) + "  fp16 zeros")
for i, s in enumerate(S_FMT):
    print(
        f"{s:4d} "
        + "".join(f"{fmt_err[name][i]:9.1e}" for name in fmt_err)
        + f"  {fp16_zeros[i]:10d}"
    )

Reading the three panels:

- **f32 and f64** behave alike: the error grows by a factor of 3 to 10
  across the sweep, with the componentwise backward error of LU, and never
  approaches κ∞·u.
- **fp16** loses accuracy as the rows enter its subnormals, with an error of
  0.8 at s = 20, where 9 entries already flush to zero. From s = 24 on it has
  no correct digits.
- **posit16 and posit32** lose digits steadily as the rows move away from 1.
  posit16 has no correct digits left from s = 36 on, and posit32 ends the
  sweep with an error of 10⁻⁴, 30 times f32's.
- **takum16 and takum32** lose digits too, but more slowly: takum16 keeps an
  error between 0.1 and 0.25 from s = 24 on. takum32 is more accurate than
  f32 up to about s = 20, and close to it after that.

For the tapered formats, then, Skeel's invariance holds for the *exact*
matrix, not for its rounded copy. Scaling rows toward 1 before storing a
matrix in a posit or takum format, the equilibration of the structure and
dynamic range notebook, is what keeps the rounding, and so the solve,
accurate.

## What to take away

- **κ(A) answers the normwise question.** For badly scaled matrices, rounding
  errors are componentwise, and κ(A)·u can overstate the error by as much as
  the scaling.
- **Skeel's cond(A) doesn't see row scaling.** In float64, the solves of the
  row-scaled systems stayed within 30u, whatever κ∞ said.
- **One step of refinement in working precision** brought LU with partial
  pivoting to a componentwise backward error below u on every tested matrix.
- **Column scaling is a change of units.** Measure each component of x
  relative to itself, and its accuracy is unchanged.
- **For posits and takums, scaling is not free.** They round entries far from
  1 more coarsely, so the stored matrix, not just the arithmetic, depends on
  the scaling.

### Where to go next

- `conditioning.ipynb`: normwise conditioning and κ·u across the formats.
- `stability.ipynb`: normwise backward error and the growth factor of LU.
- `structure_range.ipynb`: precision against magnitude for each format.